In [1]:
import pandas as pd
from sqlalchemy import create_engine

USER = "root"
PASSWORD = ""
HOST = "localhost"
PORT = 3306
DATABASE = "olist_db"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [ ]:
customers = pd.read_sql("SELECT * FROM olist_customers", engine)

orders = pd.read_sql("SELECT * FROM olist_orders", engine)

order_items = pd.read_sql("SELECT * FROM olist_order_items", engine)

payments = pd.read_sql("SELECT * FROM olist_order_payments", engine)

reviews = pd.read_sql("SELECT * FROM olist_order_reviews", engine)

products = pd.read_sql("SELECT * FROM olist_products", engine)

sellers = pd.read_sql("SELECT * FROM olist_sellers", engine)

geolocation = pd.read_sql("SELECT * FROM olist_geolocation", engine)

category_translation = pd.read_sql(
    "SELECT * FROM product_category_name_translation", engine
)

In [ ]:
print("Customers:", len(customers))
print("Orders:", len(orders))
print("Order Items:", len(order_items))
print("Payments:", len(payments))
print("Reviews:", len(reviews))
print("Products:", len(products))
print("Sellers:", len(sellers))
print("Geolocation:", len(geolocation))
print("Category Translation:", len(category_translation))

Customers: 99441
Orders: 99441
Order Items: 112650
Payments: 103886
Reviews: 99223
Products: 32951
Sellers: 3095
Geolocation: 1000163
Category Translation: 71


In [ ]:
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())

print("Duplicate order_id:", orders["order_id"].duplicated().sum())

print("Duplicate product_id:", products["product_id"].duplicated().sum())

print("Duplicate seller_id:", sellers["seller_id"].duplicated().sum())

Duplicate customer_id: 0
Duplicate order_id: 0
Duplicate product_id: 0
Duplicate seller_id: 0


In [ ]:
print(
    "Duplicate (order_id, order_item_id):",
    order_items.duplicated(subset=["order_id", "order_item_id"]).sum(),
)

Duplicate (order_id, order_item_id): 0


In [ ]:
items_agg = (
    order_items.groupby("order_id")
    .agg(
        number_of_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
    .reset_index()
)

In [ ]:
payments_agg = (
    payments.groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
    )
    .reset_index()
)

In [ ]:
ml_table = orders.merge(customers, on="customer_id", how="left")

ml_table = ml_table.merge(items_agg, on="order_id", how="left")

ml_table = ml_table.merge(payments_agg, on="order_id", how="left")

In [ ]:
print("ML table shape:", ml_table.shape)

print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

ML table shape: (99441, 17)
Duplicate order_id: 0


In [ ]:
import os

os.makedirs("../artifacts", exist_ok=True)

ml_table.to_csv("../artifacts/ml_table.csv", index=False)

print("ML table saved successfully!")

ML table saved successfully!
